# Original Code 

In [ ]:
import os

# GeoJSON
import json
from shapely.geometry import shape
from shapely.geometry.polygon import Polygon
from PIL import Image, ImageDraw

import numpy as np
from imgaug import augmenters as iaa 

import sys 
sys.path.append(os.path.abspath(".."))

from hover_net.dataloader.augmentation import (add_to_brightness,
                                               add_to_contrast, add_to_hue,
                                               add_to_saturation,
                                               gaussian_blur, median_blur)
from hover_net.dataloader.preprocessing import cropping_center, gen_targets

from hover_net.datasets.hover_dataset import HoVerDatasetBase


In [ ]:
class CoNSePDataset(HoVerDatasetBase):
    """Data Loader. Loads images from a file list and
    performs augmentation with the albumentation library.
    After augmentation, horizontal and vertical maps are
    generated.
    Args:
        file_list: list of filenames to load
        input_shape: shape of the input [h,w] - defined in config.py
        mask_shape: shape of the output [h,w] - defined in config.py
        mode: 'train' or 'valid'

    """

    def __init__(
        self,
        data_path,
        with_type=False,
        input_shape=None,
        mask_shape=None,
        run_mode="train",
        setup_augmentor=True,
    ):
        assert input_shape is not None and mask_shape is not None
        self.run_mode = run_mode
        # self.image_dir = image_dir
        # self.geojson_dir = geojson_dir

        self.image_dir = 'data/01_training_dataset_tif_ROIs'
        self.geojson_dir = 'data/01_training_dataset_geojson_nuclei'

        self.images     = os.listdir(self.image_dir)
        self.geojsons   = os.listdir(self.geojson_dir)

        # Polygon class labels
        self.classes = {
            'nuclei_tumor': 0,  # Tumor
            'nuclei_lymphocyte': 1,  # TIL
            'nuclei_plasma_cell': 1,  # TIL
            'nuclei_endothelium': 2,  # Other
            'nuclei_apoptosis': 2,  # Other
            'nuclei_stroma': 2,  # Other
            'nuclei_histiocyte': 2,  # Other
            'nuclei_melanophage': 2,  # Other
            'nuclei_neutrophil': 2,  # Other
            'nuclei_epithelium': 2,  # Other
        }

        self.with_type = with_type
        self.mask_shape = mask_shape
        self.input_shape = input_shape
        self.id = 0
        if setup_augmentor:
            self.setup_augmentor(0, 0)
        return

    def setup_augmentor(self, worker_id, seed):
        self.augmentor = self.__get_augmentation(self.run_mode, seed)
        self.shape_augs = iaa.Sequential(self.augmentor[0])
        self.input_augs = iaa.Sequential(self.augmentor[1])
        self.id = self.id + worker_id
        return

    def load_data(self, idx):
        # Load image
        img_path = os.path.join(self.image_dir, self.images[idx])
        img = Image.open(img_path)
        width, height = img.size

        # Create blank images to draw instances and annotations
        instances   = Image.new(mode="L", size=(width, height))
        annotations = Image.new(mode="L", size=(width, height))
        inst_draw = ImageDraw.Draw(instances)
        ann_draw = ImageDraw.Draw(annotations)

        # Load GeoJSON
        json_path = os.path.join(self.geojson_dir, self.geojsons[idx])
        with open(json_path, encoding="utf-8") as f:
            geojson = json.load(f)

        inst_id = 1
        for feature in geojson["features"]:
            geometry = shape(feature["geometry"])
            label = feature["properties"]["classification"]["name"]

            if geometry.geom_type == "Polygon":
                coords = geometry.exterior.coords
                if label == "nuclei_tumor":
                    ann_draw.polygon(coords, outline=1, fill=1)
                elif label in ["nuclei_lymphocyte", "nuclei_plasma_cell"]:
                    ann_draw.polygon(coords, outline=2, fill=2)
                else:
                    ann_draw.polygon(coords, outline=3, fill=3)
                inst_draw.polygon(coords, outline=inst_id, fill=inst_id)
                inst_id += 1

        # Combine instances and annotations
        ann = np.stack([np.array(instances), np.array(annotations)], axis=0).astype("int32")
        ann = np.transpose(ann, (1, 2, 0))
        img = np.array(img).astype("uint8")[:, :, :3]

        print(ann.shape)
        print(img.shape)

        return img, ann

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, ann = self.load_data(idx)

        if self.shape_augs is not None:
            shape_augs = self.shape_augs.to_deterministic()
            img = shape_augs.augment_image(img)
            ann = shape_augs.augment_image(ann)

        if self.input_augs is not None:
            input_augs = self.input_augs.to_deterministic()
            img = input_augs.augment_image(img)

        img = cropping_center(img, self.input_shape)
        feed_dict = {"img": img}

        inst_map = ann[..., 0]  # HW1 -> HW
        if self.with_type:
            type_map = (ann[..., 1]).copy()
            type_map = cropping_center(type_map, self.mask_shape)
            feed_dict["tp_map"] = type_map

        target_dict = gen_targets(inst_map, self.mask_shape)
        feed_dict.update(target_dict)

        return feed_dict

    def __get_augmentation(self, mode, rng):
        if mode == "train":
            shape_augs = [
                # * order = ``0`` -> ``cv2.INTER_NEAREST``
                # * order = ``1`` -> ``cv2.INTER_LINEAR``
                # * order = ``2`` -> ``cv2.INTER_CUBIC``
                # * order = ``3`` -> ``cv2.INTER_CUBIC``
                # * order = ``4`` -> ``cv2.INTER_CUBIC``
                # ! for pannuke v0, no rotation or translation,
                # ! just flip to avoid mirror padding
                # iaa.Affine(
                #     # scale images to 80-120% of their size,
                #     # individually per axis
                #     scale={"x": (0.8, 1.2), "y": (0.8, 1.2)},
                #     # translate by -A to +A percent (per axis)
                #     translate_percent={
                #         "x": (-0.01, 0.01), "y": (-0.01, 0.01)},
                #     shear=(-5, 5),  # shear by -5 to +5 degrees
                #     rotate=(-179, 179),  # rotate by -179 to +179 degrees
                #     order=0,  # use nearest neighbour
                #     backend="cv2",  # opencv for fast processing
                #     seed=rng,
                # ),
                # set position to 'center' for center crop
                # else 'uniform' for random crop
                iaa.CropToFixedSize(
                    self.input_shape[0], self.input_shape[1], position="center"
                ),
                iaa.Fliplr(0.5, seed=rng),
                iaa.Flipud(0.5, seed=rng),
            ]

            input_augs = [
                iaa.OneOf(
                    [
                        iaa.Lambda(
                            seed=rng,
                            func_images=lambda *args: gaussian_blur(
                                *args, max_ksize=3
                            ),
                        ),
                        iaa.Lambda(
                            seed=rng,
                            func_images=lambda *args: median_blur(
                                *args, max_ksize=3
                            ),
                        ),
                        iaa.AdditiveGaussianNoise(
                            loc=0, scale=(0.0, 0.05 * 255), per_channel=0.5
                        ),
                    ]
                ),
                iaa.Sequential(
                    [
                        iaa.Lambda(
                            seed=rng,
                            func_images=lambda *args: add_to_hue(
                                *args, range=(-8, 8)
                            ),
                        ),
                        iaa.Lambda(
                            seed=rng,
                            func_images=lambda *args: add_to_saturation(
                                *args, range=(-0.2, 0.2)
                            ),
                        ),
                        iaa.Lambda(
                            seed=rng,
                            func_images=lambda *args: add_to_brightness(
                                *args, range=(-26, 26)
                            ),
                        ),
                        iaa.Lambda(
                            seed=rng,
                            func_images=lambda *args: add_to_contrast(
                                *args, range=(0.75, 1.25)
                            ),
                        ),
                    ],
                    random_order=True,
                ),
            ]
        else:
            shape_augs = [
                # set position to 'center' for center crop
                # else 'uniform' for random crop
                iaa.CropToFixedSize(
                    self.input_shape[0], self.input_shape[1], position="center"
                )
            ]
            input_augs = []

        return shape_augs, input_augs


# Modified Code

In [ ]:
import os

# GeoJSON
import json
from shapely.geometry import shape
from shapely.geometry.polygon import Polygon
from PIL import Image, ImageDraw

import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

import sys
sys.path.append(os.path.abspath(".."))

from hover_net.dataloader.augmentation import (
    add_to_brightness,
    add_to_contrast,
    add_to_hue,
    add_to_saturation,
    gaussian_blur,
    median_blur,
)
from hover_net.dataloader.preprocessing import cropping_center, gen_targets

from hover_net.datasets.hover_dataset import HoVerDatasetBase

In [ ]:
class CoNSePDataset(HoVerDatasetBase):
    """Data Loader using Albumentations for augmentations."""

    def __init__(
        self,
        data_path,
        with_type=False,
        input_shape=None,
        mask_shape=None,
        run_mode="train",
        setup_augmentor=True,
    ):
        assert input_shape is not None and mask_shape is not None
        self.run_mode = run_mode

        self.image_dir = 'data/01_training_dataset_tif_ROIs'
        self.geojson_dir = 'data/01_training_dataset_geojson_nuclei'

        self.images = os.listdir(self.image_dir)
        self.geojsons = os.listdir(self.geojson_dir)

        self.classes = {
            'nuclei_tumor': 0,  # Tumor
            'nuclei_lymphocyte': 1,  # TIL
            'nuclei_plasma_cell': 1,  # TIL
            'nuclei_endothelium': 2,  # Other
            'nuclei_apoptosis': 2,  # Other
            'nuclei_stroma': 2,  # Other
            'nuclei_histiocyte': 2,  # Other
            'nuclei_melanophage': 2,  # Other
            'nuclei_neutrophil': 2,  # Other
            'nuclei_epithelium': 2,  # Other
        }

        self.with_type = with_type
        self.mask_shape = mask_shape
        self.input_shape = input_shape
        if setup_augmentor:
            self.setup_augmentor(run_mode)

    def setup_augmentor(self, mode):
        self.augmentor = self.__get_augmentation(mode)

    def load_data(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        img = Image.open(img_path)
        width, height = img.size

        instances = Image.new(mode="L", size=(width, height))
        annotations = Image.new(mode="L", size=(width, height))
        inst_draw = ImageDraw.Draw(instances)
        ann_draw = ImageDraw.Draw(annotations)

        json_path = os.path.join(self.geojson_dir, self.geojsons[idx])
        with open(json_path, encoding="utf-8") as f:
            geojson = json.load(f)

        inst_id = 1
        for feature in geojson["features"]:
            geometry = shape(feature["geometry"])
            label = feature["properties"]["classification"]["name"]

            if geometry.geom_type == "Polygon":
                coords = geometry.exterior.coords
                if label == "nuclei_tumor":
                    ann_draw.polygon(coords, outline=1, fill=1)
                elif label in ["nuclei_lymphocyte", "nuclei_plasma_cell"]:
                    ann_draw.polygon(coords, outline=2, fill=2)
                else:
                    ann_draw.polygon(coords, outline=3, fill=3)
                inst_draw.polygon(coords, outline=inst_id, fill=inst_id)
                inst_id += 1

        ann = np.stack([np.array(instances), np.array(annotations)], axis=0).astype("int32")
        ann = np.transpose(ann, (1, 2, 0))
        img = np.array(img).astype("uint8")[:, :, :3]

        return img, ann

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, ann = self.load_data(idx)

        if self.augmentor is not None:
            augmented = self.augmentor(image=img, mask=ann)
            img = augmented["image"]
            ann = augmented["mask"]

        img = cropping_center(img, self.input_shape)
        feed_dict = {"img": img}

        inst_map = ann[..., 0]
        if self.with_type:
            type_map = (ann[..., 1]).copy()
            type_map = cropping_center(type_map, self.mask_shape)
            feed_dict["tp_map"] = type_map

        target_dict = gen_targets(inst_map, self.mask_shape)
        feed_dict.update(target_dict)

        return feed_dict

    def __get_augmentation(self, mode):
        if mode == "train":
            aug = A.Compose(
                [
                    A.RandomCrop(height=self.input_shape[0], width=self.input_shape[1]),
                    A.HorizontalFlip(p=0.5),
                    A.VerticalFlip(p=0.5),
                    A.OneOf(
                        [
                            A.GaussianBlur(blur_limit=(3, 3), p=0.5),
                            A.MedianBlur(blur_limit=3, p=0.5),
                            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
                        ],
                        p=0.5,
                    ),
                    A.OneOf(
                        [
                            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=8, val_shift_limit=8, p=0.5),
                            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                        ],
                        p=0.5,
                    ),
                ]
            )
        else:
            aug = A.Compose(
                [
                    A.CenterCrop(height=self.input_shape[0], width=self.input_shape[1]),
                ]
            )
        return aug


# Test 

In [ ]:
# Sample Image: Replace with your test image path
test_image_path = "../data/01_training_dataset_tif_ROIs/training_set_metastatic_roi_001.tif"
try:
    test_image = Image.open(test_image_path)
    test_image = np.array(test_image)
    print(f"Image loaded successfully with Pillow, shape: {test_image.shape}")
except Exception as e:
    print(f"Failed to load image with Pillow: {e}")
    raise

if test_image.shape[-1] == 4:  # Check if image has an alpha channel
    test_image = test_image[..., :3]  # Remove the alpha channel
    print("Alpha channel removed from the image.")


# Sample Annotation (dummy mask, same size as test image)
test_mask = np.zeros(test_image.shape[:2], dtype=np.uint8)

# ---- Old Augmentation (imgaug) ---- #
def old_augmentation(img, mask):
    shape_augs = iaa.Sequential([
        iaa.CropToFixedSize(256, 256, position="center"),
        iaa.Fliplr(0.5),
        iaa.Flipud(0.5),
    ])
    input_augs = iaa.Sequential([
        iaa.OneOf([
            iaa.GaussianBlur(sigma=(0.0, 3.0)),
            iaa.MedianBlur(k=(3, 5)),
            iaa.AdditiveGaussianNoise(scale=(0, 0.05 * 255)),
        ]),
        iaa.Sequential([
            iaa.AddToHue((-8, 8)),
            iaa.AddToSaturation((-20, 20)),
            iaa.AddToBrightness((-26, 26)),
            iaa.LinearContrast((0.75, 1.25)),
        ], random_order=True)
    ])
    
    img_aug = shape_augs.augment_image(img)
    mask_aug = shape_augs.augment_image(mask)
    img_aug = input_augs.augment_image(img_aug)
    
    return img_aug, mask_aug

# ---- New Augmentation (albumentations) ---- #
def new_augmentation(img, mask):
    aug = A.Compose([
        A.RandomCrop(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 3), p=0.5),
            A.MedianBlur(blur_limit=3, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
        ], p=0.5),
        A.OneOf([
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=8, val_shift_limit=8, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        ], p=0.5),
    ])
    
    augmented = aug(image=img, mask=mask)
    return augmented["image"], augmented["mask"]

# Apply Augmentations
old_img_aug, old_mask_aug = old_augmentation(test_image, test_mask)
new_img_aug, new_mask_aug = new_augmentation(test_image, test_mask)

# ---- Visualization ---- #
def visualize_comparison(original, old_aug, new_aug, title1, title2):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original)
    axes[0].set_title("Original Image")
    axes[1].imshow(old_aug)
    axes[1].set_title(title1)
    axes[2].imshow(new_aug)
    axes[2].set_title(title2)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

visualize_comparison(
    test_image,
    old_img_aug,
    new_img_aug,
    "Old Augmentation (imgaug)",
    "New Augmentation (albumentations)"
)


In [ ]:
import os
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt

# Load Image Path
test_image_path = "../data/01_training_dataset_tif_ROIs/training_set_metastatic_roi_001.tif"

# Check if file exists
if not os.path.exists(test_image_path):
    raise FileNotFoundError(f"File not found: {test_image_path}")

# Load with Pillow
try:
    test_image = Image.open(test_image_path)
    test_image = np.array(test_image)
    print(f"Image loaded successfully, shape: {test_image.shape}")
except Exception as e:
    raise RuntimeError(f"Failed to load image with Pillow: {e}")

# Generate a dummy mask of the same size
test_mask = np.zeros(test_image.shape[:2], dtype=np.uint8)

# Visualize the image
plt.imshow(test_image, cmap="gray" if len(test_image.shape) == 2 else None)
plt.title("Loaded Image")
plt.axis("off")
plt.show()

# Now you can proceed with augmentations...
